# SentinelTex Image Analysis: YOLOv8 training

Trains the fabric defect detector on `sentineltex_dataset_v1` (6 classes: stain, spot, discoloration, tear, loose_thread, pattern_defect).

**Before running:** Settings on the right → Accelerator → **GPU T4 x2** (or P100), Internet → **On**. Add the dataset with **Add Input** (the uploaded `sentineltex_dataset_v1` dataset). Then **Run All**. Training takes about 20 to 40 minutes.

At the end, download `sentineltex_yolo_results.zip` from the Output panel. It holds `best.pt` (goes into `SentinelTex\models\`) and the evaluation numbers for the report.

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| GPU:', torch.cuda.is_available())

## 1. Find the dataset and copy it to a writable folder

In [ ]:
import shutil, glob, os, yaml
found = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
assert found, 'data.yaml not found: add the sentineltex_dataset_v1 dataset with Add Input'
src = os.path.dirname(found[0])
DATA = '/kaggle/working/dataset'
if not os.path.exists(DATA):
    shutil.copytree(src, DATA, ignore=shutil.ignore_patterns('previews', 'source_code'))
cfg = yaml.safe_load(open(f'{DATA}/data.yaml'))
cfg['path'] = DATA
yaml.safe_dump(cfg, open(f'{DATA}/data.yaml', 'w'), sort_keys=False)
for s in ['train', 'val', 'test']:
    print(s, len(os.listdir(f'{DATA}/images/{s}')), 'images')
print(cfg['names'])

## 2. Train

YOLOv8s pretrained on COCO, 640 px. Training stops early if validation mAP does not improve for 30 epochs. Seed fixed so the run can be repeated.

In [ ]:
from ultralytics import YOLO
EPOCHS = 120
model = YOLO('yolov8s.pt')
model.train(data=f'{DATA}/data.yaml', epochs=EPOCHS, imgsz=640, batch=16, patience=30,
            seed=0, deterministic=True, project='/kaggle/working/runs', name='sentineltex', exist_ok=True,
            fliplr=0.5, flipud=0.5, degrees=10, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, mosaic=1.0,
            plots=True, verbose=False)
BEST = '/kaggle/working/runs/sentineltex/weights/best.pt'
print('saved', BEST)

## 3. Evaluate on validation and on the held out test set

The test set has only real photos and was never used in training. It contains no discoloration or pattern_defect examples, so those two classes are measured on validation only. Report both tables honestly.

In [ ]:
import json
best = YOLO(BEST)
names = best.names
results = {}
for split in ['val', 'test']:
    m = best.val(data=f'{DATA}/data.yaml', split=split, imgsz=640, conf=0.001, iou=0.6,
                 project='/kaggle/working/runs', name=f'eval_{split}', exist_ok=True, plots=True, verbose=False)
    per_class = {}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        per_class[names[int(c)]] = {'precision': round(float(p), 3), 'recall': round(float(r), 3),
                                    'mAP50': round(float(ap50), 3), 'mAP50_95': round(float(ap), 3)}
    results[split] = {'precision': round(float(m.box.mp), 3), 'recall': round(float(m.box.mr), 3),
                      'mAP50': round(float(m.box.map50), 3), 'mAP50_95': round(float(m.box.map), 3),
                      'per_class': per_class}
    print(f'\n=== {split} ===')
    print(f"overall  P {results[split]['precision']}  R {results[split]['recall']}  mAP50 {results[split]['mAP50']}  mAP50-95 {results[split]['mAP50_95']}")
    for k, v in per_class.items():
        print(f"{k:15s} P {v['precision']}  R {v['recall']}  mAP50 {v['mAP50']}")
json.dump(results, open('/kaggle/working/metrics.json', 'w'), indent=2)

## 4. Image level check: does the model flag defective photos and leave clean photos alone?

This matches how SentinelTex uses the model (a sample is flagged if any defect is found).

In [ ]:
import glob, os
CONF = 0.25
tp = fp = tn = fn = 0
for img in sorted(glob.glob(f'{DATA}/images/test/*')):
    lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    truth = os.path.exists(lbl) and open(lbl).read().strip() != ''
    pred = len(best.predict(img, conf=CONF, verbose=False)[0].boxes) > 0
    tp += truth and pred; fn += truth and not pred; fp += pred and not truth; tn += not truth and not pred
image_level = {'conf': CONF, 'defect_found_correctly': tp, 'defect_missed': fn,
               'clean_flagged_wrongly': fp, 'clean_passed_correctly': tn}
print(image_level)
results['test_image_level'] = image_level
json.dump(results, open('/kaggle/working/metrics.json', 'w'), indent=2)

## 5. Save example predictions and package the results

In [ ]:
os.makedirs('/kaggle/working/examples', exist_ok=True)
for img in sorted(glob.glob(f'{DATA}/images/test/*'))[:12]:
    best.predict(img, conf=CONF, save=True, project='/kaggle/working/examples', name='pred', exist_ok=True, verbose=False)
out = '/kaggle/working/package'
os.makedirs(out, exist_ok=True)
shutil.copy(BEST, f'{out}/best.pt')
shutil.copy('/kaggle/working/metrics.json', out)
for f in ['results.csv', 'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png', 'labels.jpg']:
    p = f'/kaggle/working/runs/sentineltex/{f}'
    if os.path.exists(p): shutil.copy(p, out)
for split in ['val', 'test']:
    p = f'/kaggle/working/runs/eval_{split}/confusion_matrix.png'
    if os.path.exists(p): shutil.copy(p, f'{out}/confusion_matrix_{split}.png')
shutil.copytree('/kaggle/working/examples/pred', f'{out}/examples', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/sentineltex_yolo_results', 'zip', out)
print('Download sentineltex_yolo_results.zip from the Output panel')